# 33 — ML methods used in the discovery-9 GBSA study (tutorial walkthrough)

> **Purpose.** Notebooks 23–30 apply machine learning to the same 9-target panel.
> This walkthrough explains — in plain language — *what* each method does,
> *why* we chose it for this problem, and *how to read the numbers*. Every
> compute cell here reproduces (in miniature) one step from the real analysis
> notebooks; the take-home messages are cross-referenced with the notebook that
> did the heavy lifting.

**Prerequisites — glossary in one screen:**

| Term | Meaning |
|---|---|
| **Panel** | The 9 targets × 30 ligands matrix — the population we score over |
| **Active** | Ligand with pchembl ≥ 5, measurable binding |
| **Decoy / inactive** | Ligand with pchembl < 5 (or `is_active = False` in the OHDS metadata) |
| **Rank** | Order a scorer imposes on ligands within one target (best-first) |
| **BEDROC** | Boltzmann-Enhanced ROC — early-recognition metric, weights the top of the list |
| **LOTO CV** | Leave-One-Target-Out cross-validation — hold out one full target at a time |
| **GBSA-locked** | The reviewer-fixed combo `igb2_di4_salt0.15_st0.0072`; our baseline scorer |
| **Claim A** | "An ML model can pick a per-target GBSA-combo that beats GBSA-locked" |
| **Claim B** | "A single MD-derived feature can rank actives better than GBSA-locked" |

**Layout of this notebook** (each §-block ends with a pointer to the real analysis notebook):

1. [The problem, the data, the scoring task](#§1)
2. [BEDROC — the metric, from scratch](#§2)
3. [The baseline — GBSA-locked, on the same 9 targets](#§3)
4. [LOTO cross-validation — what it does, why](#§4)
5. [**Claim A test** — ML combo selection (NB 23 & 24)](#§5)
6. [**Claim B test** — single feature ranker (NB 28 & 29)](#§6)
7. [**Leakage tiers** — the GBSA surrogate story (NB 27)](#§7)
8. [Family stratification (NB 30)](#§8)
9. [Take-home — the clean story](#§9)

> **Reader guide.** *Aux — pedagogical walkthrough of the ML methods used in Experiment A3
> (BEDROC α=20, LOTO CV, leakage tiers, Cohen d vs Cliff's δ effect sizes).* Not a study
> result; a tutorial for reviewers unfamiliar with the metric conventions.

In [ ]:
# Standard repo-relative setup (same pattern as every other notebook here)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

from gbsabench.paths import DERIVED, FIGURES, RAW
from gbsabench.style import NAVY, GOLD, GREY, GREY_DASH, apply_style
from gbsabench import metrics
apply_style()

print(f"RAW     : {RAW}")
print(f"DERIVED : {DERIVED}")
print(f"FIGURES : {FIGURES}")

<a id='§1'></a>
## §1 — The problem, the data, the scoring task

**What we're trying to do.** For each of 9 protein targets we have 30 candidate
ligands. Some bind well (measured `pchembl` ≥ 5, `is_active = True`), most don't.
Given a *scorer* — anything that produces a number per (target, ligand) — we ask:

> *Does this scorer rank the true binders near the top of each target's list?*

The scorers we compare:

1. **Docking score** (cheap; the fastest "physics" — pose+score in seconds)
2. **GBSA ΔG** — MM/GBSA free energy from a 30 ns MD trajectory; slower, richer
3. **Single MD-derived feature** — e.g. the standard deviation of buried SASA
4. **A trained ML model** — takes any subset of the above + ligand chemistry, learns to rank

Let's look at the raw data first.

In [ ]:
# The OHDS reference: one row per (target, ligand) complex
meta = pd.read_csv(RAW / "metadata.csv")
print(f"metadata.csv: {len(meta)} rows (complexes)")
meta.head()

In [ ]:
# Class balance per target — how many actives vs decoys per target?
bal = meta.groupby("target").agg(
    total=("is_active", "size"),
    n_active=("is_active", "sum"),
)
bal["n_decoy"] = bal["total"] - bal["n_active"]
bal["pct_active"] = (100 * bal["n_active"] / bal["total"]).round(1)
display(bal)

print(f"\nAcross all 9 targets: {bal['n_active'].sum()} actives, {bal['n_decoy'].sum()} decoys")
print("Notice the imbalance: typically 5–15 actives out of 30 ligands per target.")
print("That imbalance is why we use BEDROC (§2), not plain accuracy.")

<a id='§2'></a>
## §2 — BEDROC α=20 — the metric, from scratch

### 2.1 Why not ROC-AUC?

A drug-discovery ranker only helps you if the **top of the list** is enriched
with true binders — you're never going to test the bottom half in the lab.
Plain ROC-AUC gives equal credit for correctly identifying an active at rank
5 or at rank 25. That's not what we want.

**BEDROC** (Boltzmann-Enhanced Discrimination of ROC, Truchon & Bayly 2007)
up-weights the top of the list exponentially. It has one parameter, **α**:

$$ \text{BEDROC}(\alpha) = \frac{\sum_{i=1}^{n_{\text{active}}} e^{-\alpha \cdot r_i / N}}{\text{normalization}} $$

where $r_i$ is the rank of active $i$ (1 = top) and $N$ is the total ligand
count. Large α ⇒ only the very top matters.

We use **α = 20** — a common early-recognition setting: an active in the top
1/α ≈ 5 % of the ranking contributes ~63 % of its possible score; below the
top 15 % its contribution is essentially zero.

### 2.2 BEDROC properties (worth internalising)

- Range: `0.0` (all actives at the bottom) → `1.0` (all actives at the very top).
- Random ranking hits ~ `n_active / N` (i.e. the base rate), *not* 0.5 as with ROC-AUC.
- Small changes at the top matter a lot; small changes at the bottom barely register.

Below we compute BEDROC by hand on a toy example so the formula stops being magic.

In [ ]:
def bedroc_from_scratch(y_true, scores, alpha=20.0):
    """Reference implementation — not for production use, use gbsabench.metrics.bedroc there.

    y_true : 1 for active, 0 for decoy
    scores : higher = predicted-more-likely-active
    """
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    N = len(y_true); n = y_true.sum()
    if n == 0 or n == N: return float("nan")
    # rank 1 = best score; ties resolved by original order (fine for tutorial)
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(N)
    ranks[order] = np.arange(1, N+1)
    ra = n / N
    # Truchon-Bayly 2007 formula
    sum_exp = np.sum(np.exp(-alpha * ranks[y_true==1] / N))
    Rr = ra * (1 - np.exp(-alpha)) / (np.exp(alpha/N) - 1)
    return sum_exp / Rr * (ra * np.sinh(alpha/2) /
                            (np.cosh(alpha/2) - np.cosh(alpha/2 - alpha*ra))) + \
           1 / (1 - np.exp(alpha*(1-ra)))

# Toy: 10 ligands, 3 actives, perfect ranking vs random
y   = [1,1,1,0,0,0,0,0,0,0]
perfect = [10,9,8,7,6,5,4,3,2,1]
random  = [7,3,9,1,8,2,10,4,6,5]
worst   = [1,2,3,4,5,6,7,8,9,10]  # actives at bottom

for label, s in [("perfect ranking", perfect),
                 ("random",           random),
                 ("worst possible",   worst)]:
    print(f"  {label:20s}  BEDROC α=20 = {bedroc_from_scratch(y, s, alpha=20):.3f}")
print("  (base rate = 0.30 → random ≈ 0.30, perfect ≈ 1.00)")

**Read the toy numbers as your intuition anchor.** In the real 9-target panel,
the base rate is roughly `≈ n_active/30 ≈ 0.15–0.5`, so a random panel-BEDROC
sits around 0.2–0.4. A serious ranker needs to be well above that.

**In the real notebooks** the metric is imported from `gbsabench.metrics.bedroc`,
which is faster and handles ties correctly. This §2 exists purely so you can
see the machinery. The next § uses the real metric to score GBSA.

<a id='§3'></a>
## §3 — The baseline: GBSA-locked on the 9-target panel

Before any ML enters the picture we need a baseline to beat. That baseline is
the reviewer-locked GBSA combo `igb2_di4_salt0.15_st0.0072`. Every ML claim in
notebooks 23–30 is phrased as *"does this beat GBSA-locked?"*.

Here we compute the per-target and panel BEDROC of GBSA-locked so we have the
number in memory.

In [ ]:
# Load raw GBSA data, keep only reviewer-locked combo, join with metadata
COMBO = "igb2_di4_salt0.15_st0.0072"
gbsa = pd.read_csv(RAW / "gbsa_dG_raw.csv")
gbsa = gbsa[gbsa["combo"] == COMBO][["complex_id", "mean_dG_kcalmol"]]

panel = meta.merge(gbsa, on="complex_id", how="inner")
# Score: predicted "binding strength" = −ΔG (more negative ΔG = stronger binder)
panel["score"] = -panel["mean_dG_kcalmol"]


def panel_bedroc(df, score_col, label_col="is_active", alpha=20.0):
    rows = []
    for t, sub in df.groupby("target"):
        y = sub[label_col].astype(int).values
        s = sub[score_col].values
        rows.append({"target": t, "bedroc": bedroc_from_scratch(y, s, alpha)})
    return pd.DataFrame(rows)

gbsa_pt = panel_bedroc(panel, "score")
gbsa_pt["panel_mean"] = gbsa_pt["bedroc"].mean()
display(gbsa_pt.round(3))
print(f"\nGBSA-locked panel mean BEDROC = {gbsa_pt['bedroc'].mean():.3f}")
print("(Every ML challenger from §5 onwards has to beat this number.)")

<a id='§4'></a>
## §4 — Leave-One-Target-Out cross-validation (LOTO CV)

### 4.1 The pitfall of a random train/test split

The obvious ML split is random: shuffle rows, keep 80 % for training, 20 % for
test. In this dataset that leaks — because ligands within one target are
structurally similar (they're the target's known binder series), the model
learns *target identity* rather than *binding physics*. Test accuracy looks
great, but it will not generalise to a truly new target.

### 4.2 The fix: LOTO CV

Hold out **one entire target** at a time. Train on the other 8 targets' rows,
predict on the held-out target, score. Repeat 9 times so every target is
the held-out one exactly once. Average the per-target BEDROCs → panel score.

**Why this matters:** the number LOTO produces is our best estimate of
"how well would this ML model work if we brought in a brand-new target next
month?". It's strictly harder than random split — and always lower.

### 4.3 Consequences you'll see downstream

- Numbers in NB 23–24 are **lower** than the same models with random-split
  CV would give. That's the point.
- Some targets have no siblings in the training set (e.g. bromodomain `4QB3`
  is the only bromodomain). Under LOTO the model *cannot* learn its family
  and often collapses to random-like performance — see NB 30.

*(No code cell here — LOTO is used inside every ML notebook that follows;
the mechanism is the same each time.)*

<a id='§5'></a>
## §5 — Claim A test — Can ML pick a better GBSA-combo per target?

*(covered by notebooks* `23_ml_combo_selection.ipynb` *and* `24_full_factorial_ml.ipynb`)*

### 5.1 The question, in plain english

We have **48 different GBSA parameter combinations** (5 physics knobs × several
settings). One of them, `igb2_di4_salt0.15_st0.0072`, is the reviewer-locked
baseline; any of the other 47 could in principle be better for some particular
target. So: given a target we've never seen, can an ML model *look at cheap
MD-derived features* and *pick the best GBSA combo for that target*?

If yes → we save 47× compute time (only run the picked combo).

### 5.2 The input features

Notebook 23 uses **41 MD descriptors** aggregated per target (median + std
over its 30 complexes). These are cheap to compute from any trajectory — no
GBSA needed. Examples:

| Feature | What it captures |
|---|---|
| `rmsd_bb_mean_A` | Protein backbone flexibility (Å) |
| `lig_buried_sasa_mean_A2` | Median ligand solvent-accessible-area buried by protein |
| `vdw_contacts_mean` | Average number of van-der-Waals contacts |
| `n_hb_mean` | Average number of hydrogen bonds |
| `ifp_tanimoto_median_vs_ref` | Interaction-fingerprint similarity to reference pose |

### 5.3 The models tried (in NB 23)

All from `sklearn`, all wrapped in a `Pipeline` with a `StandardScaler` +
`SimpleImputer(strategy='median')` so features on different units don't
hijack the fit:

- **Ridge**: linear regression with L2 penalty. Interpretable, cheap.
- **Random Forest**, **Extra Trees**: tree ensembles, capture non-linear cutoffs.
- **HistGradientBoosting**: modern GBDT, best speed / accuracy usually.
- **SVM-RBF**: kernel method, catches smooth non-linear boundaries.
- **ElasticNet**: L1 + L2, does implicit feature selection.

### 5.4 The verdict

Every model was scored under LOTO CV with a bootstrap 95 % confidence interval
(B = 1000 resamples, seed 20250901) and a permutation p-value versus
"random combo pick" (B = 1000).

**Ridge won the ML tournament** with a panel BEDROC of **0.511 [0.334, 0.684]**,
permutation p = 0.081. But **GBSA-locked scores 0.541** — and the lower bound
of Ridge's 95 % CI (0.334) is *far* below that. No model's CI lower bound
crosses the baseline.

> **Conclusion (NB 23):** ML cannot pick a per-target GBSA-combo that reliably
> beats fixing one physics recipe for the whole panel. "Claim A rejected."

### 5.5 The "degeneracy trap" (NB 24)

Notebook 24 goes bigger: **~4000 rows** (199 complexes × 20 combos), features
= per-complex MD + per-target aggregates + the combo's own knobs. A
`HistGradientBoostingClassifier(max_depth=4)` is trained to predict
`is_active` under Leave-One-Target-Out.

**Trap:** the model achieves panel BEDROC 0.562 vs GBSA-locked 0.440.
Looks like a win. Then the sanity check: *which combo does the "picker"
recommend for each held-out target?* Answer: **the same combo, 100 % of the
time** (`igb1_di1_salt0.15_st0`).

That's not combo-selection intelligence — it's a *constant recommendation*.
The 0.562 measures intra-target ranking under a single fixed combo, which is
a different question (and one we already have a baseline for).

In [ ]:
# Reproduce the headline numbers from NB 23-24 (from checked-in derived tables
# if present; else compute on the fly)
sr = DERIVED / "study2"  # NB 09-related; NB 23 writes to DERIVED / "deep_research_wide.csv"
wide = DERIVED / "deep_research_wide.csv"
if wide.exists():
    dw = pd.read_csv(wide)
    print(f"deep_research_wide.csv: {len(dw)} rows")
    pick = dw[(dw['protocol'] == 'P1') & dw['model'].isin(['GBSA-locked', 'Ridge', 'RandomForest', 'SVM-RBF', 'HistGB'])]
    if len(pick):
        display(pick[['model','panel_bedroc','ci_low','ci_high','perm_p']].round(3))
else:
    print("deep_research_wide.csv not present — run NB 23 first to regenerate.")
    print("Headline numbers from the NB 23 markdown:")
    print("  GBSA-locked   panel BEDROC = 0.541  [0.36, 0.71]")
    print("  Ridge         panel BEDROC = 0.511  [0.334, 0.684]  perm p = 0.081")
    print("  SVM-RBF       panel BEDROC = 0.512")
    print("  HistGB        panel BEDROC = 0.454")

<a id='§6'></a>
## §6 — Claim B test — Can a single MD feature beat GBSA?

*(covered by notebooks* `28_single_feature_bedroc.ipynb` *and* `29_rank_fusion_deployable.ipynb`)*

### 6.1 The claim, as first reported

Notebook 28 tested every single MD feature as a *panel scorer*: for each
feature and each direction (higher = active vs lower = active), compute
per-target BEDROC and average. The winner:

> `lig_buried_sasa_std_A2` (fluctuation of buried SASA over the trajectory)
> gave panel BEDROC = **0.674** vs GBSA-locked 0.609.

**"A one-number scorer beats the physics!"** That's Claim B.

### 6.2 Then the hardening tests…

The claim was subjected to 7 robustness checks:

1. **Canonical 9-target panel** (with 4A5S recovered as the 9th target):
   feature = 0.603 [0.39, 0.78] vs GBSA = 0.541 [0.36, 0.71]. **CIs overlap.**
2. **Residualise the score on ligand MW** (in case the feature was just
   picking up ligand size): drops to 0.506 (below GBSA).
3. **Bound-frames only** (exclude solvent-detached snapshots): 0.463 (below
   random!).
4. **Tighter active label pchembl ≥ 7**: 0.978 on a panel where only 3 of 8
   targets have decoys → degenerate panel, not a real signal.
5. **PBC-cleaned trajectories**: essentially unchanged.
6. **Alt-label 6**: similar drift.

> **Conclusion (NB 28):** Claim B was an artifact of the 8-target subset that
> excluded 4A5S. On the canonical 9-target panel the single feature is not
> distinguishable from GBSA. **Claim B retracted.**

### 6.3 What NB 29 adds (rank fusion)

Even if one feature isn't enough, maybe an **average of the top-K features'
ranks** is. NB 29 does exactly this: on training targets pick the top-K
feature-directions by panel BEDROC, on the held-out target average their
normalised ranks, score. K swept over {1, 2, 3, 5, 8, 10, 15, 20, 30}.

Best K = 3 → panel BEDROC 0.508 (still under GBSA). Above K = 15 dilution
makes it worse. Same target is the "best pick" in 5/9 folds:
`lig_buried_sasa_std_A2` — the same feature that failed the hardening tests
in NB 28.

**No trick reliably beats GBSA on the 9-target panel.**

<a id='§7'></a>
## §7 — Leakage tiers — the GBSA surrogate story (NB 27)

This is the single most important pedagogical notebook in the ML set —
*this is what you learn from more than anything else*.

### 7.1 The question

Can we predict GBSA ΔG from cheaper descriptors? If yes, we skip the 30 ns MD
and cheap-scan a million ligands. Two candidate feature sets:

- **MD descriptors**: 60 numbers computed from the trajectory
  (radii of gyration, SASA, hbonds, VDW contacts, dipoles, salt bridges…)
- **Ligand chemistry**: 7 RDKit descriptors that need no MD at all
  (MW, TPSA, LogP, HBA/HBD, heavy-atom count, rotatable bonds)

### 7.2 The trap of "looking good"

If you naively throw all 60 MD features into a `HistGradientBoostingRegressor`
and score by per-target Pearson r under LOTO you get:

> **Tier 1 (quasi-circular):** Pearson r ≈ **0.93** across targets.

Beautiful. Published. Except… several MD features implicitly *encode* the
GBSA integrand:

- `lig_coulomb_mean` — Coulomb sum over the trajectory
- `sum_ligprot_vdw_mean` — VDW sum
- `lig_dipole_mean` — dipole moment (feeds directly into GB solvation)

Feeding these to a regressor that predicts ΔG is training on the answer.

### 7.3 Removing the obviously-leaking features → Tier 2

Drop Coulomb, VDW sums, dipoles, everything that is itself an energy or a
molecular-mechanics scalar. Keep only geometry + counts + ligand-chem:

> **Tier 2 (partial-leak):** Pearson r ≈ **0.778**

Still surprisingly high. Why? Because MD-derived geometry (buried SASA, RMSD,
pocket volume) is *proxy-correlated* with GBSA — a stable, well-fitting ligand
with lots of contact has both a low ΔG and specific geometric signatures.

### 7.4 The truly clean tier — ligand-only

Drop *all* MD features. Use only the 7 ligand chemistry descriptors that you
could compute from a SMILES without ever running MD:

> **Tier 3 (deployable):** Pearson r ≈ **0.455 [0.236, 0.674]** (LOTO, bootstrap)

That is the honest, deployable number. It says: *SMILES-only ML can explain
about 20 % of GBSA ΔG variance (r² ≈ 0.21) across targets*. Useful for a
coarse pre-screen (top 20 % by predicted ΔG probably includes most of the
true top 40 %), not a replacement for GBSA.

### 7.5 The pedagogical lesson

> **When your ML model looks too good on your own dataset, you have leakage.
> Always define what "deployable" means and enforce it in your feature set.**

The 0.93 → 0.78 → 0.46 collapse is the story.

In [ ]:
# Illustrate the leakage-tier collapse from NB 27 as a bar chart
tiers = pd.DataFrame([
    ("Tier 1  (all 60 MD features — quasi-circular)", 0.930, "—"),
    ("Tier 2  (MD-honest, no MM-energy features)",    0.778, "—"),
    ("Tier 3  (ligand-chem only, deployable)",         0.455, "[0.236, 0.674]"),
], columns=["tier", "pearson_r", "95%_CI"])
display(tiers)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(range(3), tiers["pearson_r"], color=[NAVY, GOLD, GREY_DASH],
        edgecolor="white")
for i, r in enumerate(tiers["pearson_r"]):
    ax.text(r + 0.01, i, f"r = {r:.2f}", va="center", fontsize=10)
ax.set_yticks(range(3))
ax.set_yticklabels(tiers["tier"], fontsize=9)
ax.axvline(0, color=GREY, lw=0.5)
ax.set_xlim(0, 1.05); ax.set_xlabel("per-target Pearson r  (LOTO)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES / "33_ml_walkthrough_tutorial_leakage_tiers.png", dpi=140, bbox_inches="tight")
plt.show()

<a id='§8'></a>
## §8 — Family stratification (NB 30)

### 8.1 The idea

9 targets is a small panel. If we grouped them by *protein family* — proteases,
kinases, bromodomains, chaperones, other — could we spot where ML wins or
loses systematically?

The panel breaks down as:

| Family | n_targets | targets |
|---|---:|---|
| Protease | 4 | 2XU3, 3I06, 4L7G, 9D9I |
| Kinase | 2 | 4A5S, 5HU9 |
| Bromodomain | 1 | 4QB3 |
| Chaperone | 1 | 8ELC |
| Other | 1 | 9SI4 |

### 8.2 The result

| Family | GBSA BEDROC | ML BEDROC (best) | comment |
|---|---:|---:|---|
| Protease (n=4) | 0.351 | ~0.35 (all four models tied) | No lift — physics is enough |
| Kinase (n=2) | 0.820 | ~0.82 | Ceiling: no headroom for ML |
| **Bromodomain (n=1)** | **0.517** | **0.132 – 0.256** | **4QB3 collapse — see below** |
| Chaperone (n=1) | 0.741 | — | Singleton, no CI |
| Other (n=1) | 0.568 | — | Singleton, no CI |

### 8.3 Why 4QB3 collapses

**Bromodomains are chemically unlike proteases or kinases.** Under LOTO the
training set contains zero bromodomains. The ML model, trained on 8 non-
bromodomain targets, has no basis for predicting on 4QB3 — it falls back to
the *training mean* for that ligand's descriptors, which is roughly random
for actives specifically.

**This is a generic ML failure mode.** Any leave-one-family-out task on a
small panel where that family is a singleton will do this.

> **Take-home:** Report per-family BEDROCs. If any family is a singleton, the
> LOTO score for it is not a real generalisation number — it is a coldstart
> penalty. Extending the panel to 18 more targets (validation set) with 3+
> more bromodomains would let us re-test this.

<a id='§9'></a>
## §9 — Take-home: the clean story to remember

After all six ML notebooks, the honest sentence is short:

> **On the 9-target discovery panel, no ML model reliably beats GBSA-locked;
> a ligand-chem-only surrogate explains ~20 % of GBSA ΔG variance (r ≈ 0.45),
> useful for coarse pre-screens but not a replacement.**

The 5 rules you should internalise from this exercise:

1. **Choose the metric that fits the task.** For imbalanced ranking →
   BEDROC α=20, not ROC-AUC. (§2)
2. **Cross-validate the way you'll deploy.** For "predict on a new target"
   → Leave-One-Target-Out, not random split. (§4)
3. **Watch for leakage tiers.** Every feature you add is a chance for the
   answer to slip into the input. Enforce a "deployable" definition. (§7)
4. **Sanity-check the picker.** An ML model that "beats baseline" but always
   picks the same class/combo is not intelligent — it is a constant
   recommendation with a fancy hat. (§5.5)
5. **Stratify on the natural grouping.** A panel-mean can hide a family that
   your model can't even see. Report per-family CIs and flag singletons. (§8)

### Next steps in the study — where the ML story could still land differently

- **18 additional newbench_27 targets (validation)** — if the discovery-9
  finding that no ML beats GBSA holds across another 18 targets, the negative
  result is publishable as-is. If a family in the 18 (e.g. 3 more bromodomains)
  behaves systematically differently, family-stratified ML may become the
  right frame.
- **Richer ligand features** (Morgan fingerprints, 3D shape) — could push
  Tier 3 above r = 0.5 without leakage.
- **Rank-fusion on chemistry-only descriptors** (analogous to NB 29 but for
  the ligand-chem tier) — untested here.